In [1]:
import pandas as pd

In [2]:
x = pd.read_csv("input/feca_annotations.tsv", sep="\t")
x

,patient_code,TubeID,sex,cancer_type.c,stage.c,ici_basel_setting.c2,icitype_fup_combo.c,timepoint,toxicity_first_date,toxicity,country,PPI,Cortisone,NSAID,Asprin,AB
0,SWE-001,192fece10000,M,Melanoma,Others,Neoadj/adj,PD-1/PD-L1,T0,NaN,no,SWE,NaN,NaN,NaN,NaN,NaN
1,SWE-001,192fece10001,M,Melanoma,Others,Neoadj/adj,PD-1/PD-L1,T1,NaN,no,SWE,NaN,NaN,NaN,NaN,NaN
2,SWE-002,192fece10003,M,Melanoma,IV,Advanced 1st,"CTLA-4,PD-1",T0,NaN,no,SWE,0.0,1.0,0.0,0.0,0.0
3,SWE-002,192fece10004,M,Melanoma,IV,Advanced 1st,"CTLA-4,PD-1",T1,NaN,no,SWE,0.0,1.0,0.0,0.0,0.0
4,SWE-003,192fece10008,M,Melanoma,IV,"Advanced>=2nd,maint","CTLA-4,PD-1",T0,08/12/2020,yes,SWE,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
430,NOR-023,192fece30049,F,Lung,IV,Advanced 1st,PD-1/PD-L1,T2,04/01/2023,yes,NOR,NaN,NaN,NaN,NaN,NaN
431,NOR-024,192fece30050,M,Lung,IV,Neoadj/adj,PD-1/PD-L1,T0,NaN,no,NOR,NaN,NaN,NaN,NaN,NaN
432,NOR-024,192fece30052,M,Lung,IV,Neoadj/adj,PD-1/PD-L1,T1,NaN,no,NOR,NaN,NaN,NaN,NaN,NaN
433,NOR-025,192fece30042,M,Lung,Others,"Advanced>=2nd,maint",PD-1/PD-L1,T0,NaN,no,NOR,NaN,NaN,NaN,NaN,NaN


In [3]:
y = pd.read_csv("input/sample_info.tsv", sep="\t")
y

,96-well,No.,Sample ID,TP,Sex,CF ID,Pep. Conc. ug/uL,250ng uL,750ng uL
0,1_A1,18,SWE001,T0,M,XCHEJF_001,0.1750,1.43,4.29
1,1_B1,19,SWE001,T1,NaN,XCHEJF_002,0.1930,1.30,3.89
2,1_C1,20,SWE002,T0,M,XCHEJF_003,0.2164,1.16,3.47
3,1_D1,21,SWE002,T1,NaN,XCHEJF_004,0.2342,1.07,3.20
4,1_E1,22,SWE004,T0,M,XCHEJF_005,0.2236,1.12,3.35
...,...,...,...,...,...,...,...,...,...
249,3_H11,243,ITA095,T0,F,XCHEJF_250,0.2536,0.99,2.96
250,3_A12,244,ITA096,T0,M,XCHEJF_251,0.2285,1.09,3.28
251,3_B12,245,ITA096,T1,NaN,XCHEJF_252,0.2545,0.98,2.95
252,3_C12,246,ITA105,T0,F,XCHEJF_253,0.2307,1.08,3.25


In [4]:
# Copy to avoid modifying originals
x_clean = x.copy()
y_clean = y.copy()

# Remove "-" from patient_code
x_clean["patient_code_clean"] = x_clean["patient_code"].str.replace("-", "", regex=False)

# Ensure Sample ID is string
y_clean["Sample ID"] = y_clean["Sample ID"].astype(str)

merged = pd.merge(
    y_clean,
    x_clean,
    left_on="Sample ID",
    right_on="patient_code_clean",
    how="inner"
)

final_df = merged[
    [
        "Sample ID",
        "TP",
        "Sex",
        "CF ID",
        "cancer_type.c",
        "stage.c",
        "toxicity_first_date",
        "toxicity",
        "country",
        "PPI",
        "Cortisone",
        "NSAID",
        "Asprin",
        "AB"
    ]
].copy()

final_df = final_df.rename(
    columns={
        "cancer_type.c": "cancer_type",
        "stage.c": "stage"
    }
)
final_df

,Sample ID,TP,Sex,CF ID,cancer_type,stage,toxicity_first_date,toxicity,country,PPI,Cortisone,NSAID,Asprin,AB
0,SWE001,T0,M,XCHEJF_001,Melanoma,Others,NaN,no,SWE,NaN,NaN,NaN,NaN,NaN
1,SWE001,T0,M,XCHEJF_001,Melanoma,Others,NaN,no,SWE,NaN,NaN,NaN,NaN,NaN
2,SWE001,T1,NaN,XCHEJF_002,Melanoma,Others,NaN,no,SWE,NaN,NaN,NaN,NaN,NaN
3,SWE001,T1,NaN,XCHEJF_002,Melanoma,Others,NaN,no,SWE,NaN,NaN,NaN,NaN,NaN
4,SWE002,T0,M,XCHEJF_003,Melanoma,IV,NaN,no,SWE,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
538,ITA087,T1,NaN,XCHEJF_246,Lung,IV,22/09/2021,yes,ITA,NaN,NaN,NaN,NaN,NaN
539,ITA087,T1,NaN,XCHEJF_246,Lung,IV,22/09/2021,yes,ITA,NaN,NaN,NaN,NaN,NaN
540,ITA087,T1,NaN,XCHEJF_246,Lung,IV,22/09/2021,yes,ITA,NaN,NaN,NaN,NaN,NaN
541,ITA096,T0,M,XCHEJF_251,Lung,Others,08/09/2021,yes,ITA,NaN,NaN,NaN,NaN,NaN


In [5]:
final_df.to_csv("input/sample_info_completed.tsv", sep="\t", index=False)

In [6]:
import pandas as pd

# Clean IDs in x (remove "-")
x_ids = (
    x["patient_code"]
    .dropna()
    .astype(str)
    .str.replace("-", "", regex=False)
)

# Clean IDs in y (keep as is, or normalize too just in case)
y_ids = (
    y["Sample ID"]
    .dropna()
    .astype(str)
    .str.replace("-", "", regex=False)  # safe to apply to both
)

# Convert to sets
ids_x = set(x_ids)
ids_y = set(y_ids)

# Intersection
common_ids = ids_x & ids_y

# Differences
only_in_x = ids_x - ids_y
only_in_y = ids_y - ids_x
only_in_y_df = pd.DataFrame({"IDs_only_in_plasma_proteomics": list(only_in_y)})

# Summary
print(f"Total IDs in metagenomics: {len(ids_x)}")
print(f"Total IDs in plasma proteomics: {len(ids_y)}")
print(f"Common IDs: {len(common_ids)}")

print(f"\nIDs only in metagenomics: {len(only_in_x)}")
print(f"IDs only in plasma proteomics: {len(only_in_y)}")

print(f"\nTotal differential IDs: {len(only_in_x) + len(only_in_y)}")

Total IDs in metagenomics: 216
Total IDs in plasma proteomics: 122
Common IDs: 111

IDs only in metagenomics: 105
IDs only in plasma proteomics: 11

Total differential IDs: 116


In [7]:
only_in_y_df

,IDs_only_in_plasma_proteomics
0,ITA095
1,ITA077
2,ITA043
3,ITA105
4,ITA079
5,SWE021
6,ITA062
7,ITA094
8,ITA061
9,ITA091
